# Task 3 — Multi-Agent Financial Research System**CDAZZDEV Senior Machine Learning Engineer Assessment**| Section | Deliverable | Marks ||---|---|---|| 3A | Single tool-using research agent with five tools, autonomous selection, observe-replan cycle, graceful failure | 50 || 3B | Two-agent pipeline with enforced tool restriction, Pydantic handoff, visible trace, critique loop | 35 || 3C | Short-term memory, persistent per-ticker cache, `agent_trace.jsonl` | 15 || Bonus | Streamlit trace dashboard | +5 |### Why LangGraph rather than CrewAIThree criteria in this task are about things being **visible and enforced**, not claimed:* *"agent decides order based on observations — not a fixed sequence"** *"at least one visible observe-and-replan cycle"** *"agents have enforced, separate tool access"*In LangGraph each is a structural property of the graph. Tool selection is a **conditionaledge** evaluated on the model's own output, so there is no sequence to hardcode. Theobserve-replan cycle *is* the `agent → tools → agent` edge — it appears in the trace becauseit is the control flow, not because we logged it. And tool restriction is enforced bybinding different tool lists to different nodes: Agent B is never handed `get_price_data`,so it cannot call it.A system prompt saying *"do not use the price tools"* is a request. Not passing the tool isa guarantee. CrewAI would be less code; this is more defensible, and Section 2 of the briefsays every part of the submission has to be defended in interview.**Graph shapes**```3A:   START → agent ⇄ tools → synthesise → END                └─ conditional: does the model want a tool? ─┘3B:   START → agent_a ⇄ tools_a → handoff → agent_b ⇄ tools_b                                                 ↓                                             critique ──no──→ final_report → END                                                 │yes                                         clarify_a ─────────────┘```

## 0 · Setup

In [ ]:
%%capture!pip install -q "langgraph>=0.2.28" "langchain-core>=0.3" "langchain-openai>=0.2" \                "openai>=1.40" "pydantic>=2.7" "yfinance>=0.2.40" ddgs pandas numpy requests

In [ ]:
import os, sys, json, logging, warningsfrom pathlib import Pathwarnings.filterwarnings("ignore")logging.basicConfig(level=logging.WARNING, format="%(levelname)-7s %(name)s | %(message)s")REPO = "CDAZZDEV-MLE-Udith"if not Path(REPO).exists() and not (Path.cwd() / "common").exists():    !git clone -q https://github.com/YOUR_USERNAME/{REPO}.git    %cd {REPO}elif Path(REPO).exists() and Path.cwd().name != REPO:    %cd {REPO}ROOT = Path.cwd()sys.path[:0] = [str(ROOT), str(ROOT / "task3_agentic" / "src"), str(ROOT / "task1_financial" / "src")]print("Repository root:", ROOT)

In [ ]:
from common.llm_client import get_secretfor name in ("GROQ_API_KEY", "OPENROUTER_API_KEY"):    v = get_secret(name)    print(f"{name:<20} {'configured' if v else 'NOT SET'}")if not (os.environ.get("GROQ_API_KEY") or os.environ.get("OPENROUTER_API_KEY")):    raise SystemExit("Add a key to Colab Secrets (key icon, left sidebar) and re-run.")

### Verify the offline components firstSchemas, tracer, cache and tool-restriction declarations are tested without touching thenetwork, so a failure here is a code bug rather than a rate limit.

In [ ]:
!python task3_agentic/tests/test_task3_offline.py

In [ ]:
TICKER = "NVDA"from tracing import AgentTracerfrom tools import ToolContext, build_tools, ALL_TOOL_NAMES, AGENT_A_TOOLS, AGENT_B_TOOLSfrom memory import ResearchCachefrom agents import build_chat_model, run_single_agent, run_multi_agentfrom common.llm_client import LLMClient, TierTRACE_PATH = ROOT / "task3_agentic" / "logs" / "agent_trace.jsonl"TRACE_PATH.parent.mkdir(parents=True, exist_ok=True)chat_model = build_chat_model(temperature=0.1)llm_client = LLMClient(tier=Tier.FAST)      # Used inside llm_sentiment only.print("Chat model :", chat_model.model_name)print("Sentiment  :", llm_client.active_model, "via", llm_client.active_provider)print("Tools      :", ALL_TOOL_NAMES)

## 1 · All five tools work (15 marks)Called directly, before any agent is involved. Each returns a structured envelope —`{ok, data, error, suggestion}` — never a bare string and never an exception into theagent loop. The `suggestion` field on failure is what the error-handling criterion restson: it gives the model an actionable alternative to re-plan around.

In [ ]:
smoke_tracer = AgentTracer(path=TRACE_PATH, echo=False)smoke_ctx = ToolContext(tracer=smoke_tracer, llm=llm_client, agent_name="smoke")tools = {t.name: t for t in build_tools(smoke_ctx)}print(f"{'Tool':<22} {'ok':<5} {'ms':>8}  Return type / sample")print("─" * 100)r1 = tools["get_price_data"].invoke({"ticker": TICKER, "period": "2y"})print(f"{'get_price_data':<22} {str(r1['ok']):<5} {smoke_tracer.records[-1].duration_ms:>8.0f}  "      f"dict[{len(r1['data'])} keys] price={r1['data']['current_price']} rsi={r1['data']['rsi_14']}")r2 = tools["get_news"].invoke({"ticker": TICKER, "n": 12})print(f"{'get_news':<22} {str(r2['ok']):<5} {smoke_tracer.records[-1].duration_ms:>8.0f}  "      f"list[{len(r2['data'])} dicts] '{r2['data'][0]['headline'][:48]}…'")r3 = tools["calculate_volatility"].invoke({"ticker": TICKER, "window": 30})print(f"{'calculate_volatility':<22} {str(r3['ok']):<5} {smoke_tracer.records[-1].duration_ms:>8.0f}  "      f"dict, vol={r3['data']['annualised_volatility_pct']}% "      f"p{r3['data']['percentile_vs_two_year']}")heads = [h['headline'] for h in r2['data'][:8]] if r2['ok'] else []r4 = tools["llm_sentiment"].invoke({"headlines": heads})print(f"{'llm_sentiment':<22} {str(r4['ok']):<5} {smoke_tracer.records[-1].duration_ms:>8.0f}  "      f"dict, score={r4['data']['aggregate_score']} ({r4['data']['label']})")r5 = tools["web_search"].invoke({"query": f"{TICKER} stock analyst outlook risks", "max_results": 5})print(f"{'web_search':<22} {str(r5['ok']):<5} {smoke_tracer.records[-1].duration_ms:>8.0f}  "      f"{'list[' + str(len(r5['data'])) + ' dicts]' if r5['ok'] else 'FAILED → ' + r5['suggestion'][:44]}")

### Graceful failure (7 marks)Deliberately break two tools and confirm each returns an actionable suggestion instead ofraising. This is the behaviour the agent depends on to re-plan — it sees the suggestion inits context and takes the alternative route.

In [ ]:
bad_ticker = tools["get_price_data"].invoke({"ticker": "NOTAREALTICKER123"})print("Invalid ticker:")print(f"  ok         : {bad_ticker['ok']}")print(f"  error      : {bad_ticker['error'][:110]}")print(f"  suggestion : {bad_ticker['suggestion'][:150]}")no_heads = tools["llm_sentiment"].invoke({"headlines": []})print("\nEmpty headline list:")print(f"  ok         : {no_heads['ok']}")print(f"  suggestion : {no_heads['suggestion'][:150]}")print("\nNeither raised. The agent receives data describing the failure, plus a route forward.")

## 2 · Task 3A — autonomous single agent (50 marks)The agent is given the query and nothing else. It decides which tools to call and in whatorder. Watch the stream below: each step prints the model's stated intention(**REASONING**), the tool it chose (**DECIDED**), and what came back (**OBSERVED**).The **observe-and-replan cycle** is the `agent → tools → agent` edge. You will see theagent call a tool, read the result, and pick its *next* tool based on what that resultactually said — for example fetching news after seeing an unusual price move, or reachingfor `web_search` after `get_news` returns thin coverage.

In [ ]:
tracer_3a = AgentTracer(path=TRACE_PATH, echo=True)ctx_3a = ToolContext(tracer=tracer_3a, llm=llm_client, agent_name="Solo")result_3a = run_single_agent(TICKER, ctx_3a, chat_model, max_iterations=8, verbose=True)

In [ ]:
print("AUTONOMY EVIDENCE")print("─" * 78)order = [r.tool for r in tracer_3a.records]print(f"Tools called, in the order the model chose : {' → '.join(order)}")print(f"Distinct tools used                        : {len(set(order))} of 5")print(f"Reasoning iterations                       : {result_3a['iterations']}")print()print("No sequence is hardcoded anywhere. The next node is chosen by a conditional edge")print("that inspects whether the model emitted tool_calls — re-running this cell will")print("frequently produce a different order, because the decision is the model's.")tracer_3a.print_summary()

## 3 · Task 3B — two agents with enforced restriction and a critique loop (35 marks)| Agent | Role | Tools | Cannot reach ||---|---|---|---|| **A — Data Analyst** | Quantitative measurement | `get_price_data`, `calculate_volatility`, `llm_sentiment` | `web_search`, `get_news` || **B — Research Writer** | Qualitative synthesis | `web_search`, `get_news` | all price/volatility tools |**The handoff is a validated `QuantBrief`**, not a string. Agent A cannot hand Agent Banything else. Two fields make it a real contract rather than decoration:* `quantitative_findings` — a validator rejects any finding that contains no figure, which  stops Agent A drifting into Agent B's job.* `data_gaps` — **required**. Agent A must declare what it could not measure. Without this  field, an agent whose volatility tool failed simply omits volatility and Agent B cannot  tell "not applicable" from "never measured". Forcing the declaration is what makes the  final report honest.**The critique loop** is model-driven: Agent B is asked whether anything blocks it, andwith Agent A obliged to declare gaps, it usually does have a question. `require_critique=True`guarantees one visible cycle for the reviewer; if it has to force one, the result is flagged`forced_critique=True` so that is never mistaken for the model's own initiative.

In [ ]:
tracer_3b = AgentTracer(path=TRACE_PATH, echo=True)result_3b = run_multi_agent(    TICKER, tracer_3b, llm_client, chat_model, require_critique=True)

### Handoff and critique evidence

In [ ]:
brief = result_3b["quant_brief"]print("STRUCTURED HANDOFF (Agent A → Agent B)")print("─" * 78)if brief:    print(f"Type on the wire : {type(brief).__name__} (Pydantic, validated at the boundary)")    print(f"Confidence       : {brief.confidence}")    print(f"Trend regime     : {brief.trend_regime.value}")    print(f"Declared gaps    : {brief.data_gaps or 'none'}")    print("\nSerialised form Agent B received:")    print(json.dumps(brief.model_dump(mode='json'), indent=2)[:1400])else:    print("Agent A failed to produce a valid brief — see the validation log above.")print("\n\nCRITIQUE LOOP")print("─" * 78)req, ans = result_3b["clarification"], result_3b["clarification_answer"]print(f"Rounds executed : {result_3b['critique_rounds']}")print(f"Forced          : {result_3b['forced_critique']}"      f"{'  (Agent B raised none itself; one was synthesised for demonstration)' if result_3b['forced_critique'] else '  (Agent B raised this on its own)'}")if req:    print(f"\nB → A  field    : {req.field_in_question}")    print(f"       question : {req.question}")    print(f"       why      : {req.why_it_matters}")if ans:    print(f"\nA → B  answer   : {ans.answer[:400]}")    print(f"       values   : {json.dumps(ans.supporting_values)[:220]}")    print(f"       could_not_answer: {ans.could_not_answer}")

In [ ]:
print("TOOL RESTRICTION — ENFORCED, NOT REQUESTED")print("─" * 78)by_agent = {}for r in tracer_3b.records:    by_agent.setdefault(r.agent, set()).add(r.tool)for agent, used in sorted(by_agent.items()):    allowed = set(AGENT_A_TOOLS) if agent == "AgentA" else set(AGENT_B_TOOLS) if agent == "AgentB" else set(ALL_TOOL_NAMES)    violations = used - allowed    print(f"{agent:<10} used {sorted(used)}")    print(f"{'':<10} allowed {sorted(allowed)}")    print(f"{'':<10} violations: {sorted(violations) or 'NONE'}")print()print("Agent B physically cannot call get_price_data: the tool object is never placed in")print("the list passed to its ToolNode, and its node rejects any call to a name outside")print("that list. Every number in the final report therefore came through the handoff.")

## 4 · Task 3C — memory and observability (15 marks)### 4.1 Short-term memory (5 marks)A follow-up question answered from context, **without re-calling any tool**. The proof isthe trace: we record the call count before and after, and it does not change.

In [ ]:
from langchain_core.messages import HumanMessagecalls_before = len(tracer_3a.records)history = result_3a["state"]["messages"]follow_up = (    "Follow-up: what was the RSI-14 reading and the 30-day annualised volatility you "    "retrieved earlier for this ticker? Answer from what you already have in this "    "conversation. Do NOT call any tool.")answer = chat_model.invoke(list(history) + [HumanMessage(content=follow_up)])calls_after = len(tracer_3a.records)print("FOLLOW-UP ANSWER (from context)")print("─" * 78)print(answer.content[:700])print("\n" + "─" * 78)print(f"Tool calls before : {calls_before}")print(f"Tool calls after  : {calls_after}")print(f"New tool calls    : {calls_after - calls_before}  "      f"{'✓ answered purely from short-term memory' if calls_after == calls_before else '✗ re-fetched'}")

There are two layers of short-term memory here, worth distinguishing in interview:1. **Conversational** — the LangGraph state carries the full message list, so prior   `ToolMessage` results stay in context and the model can read them back.2. **Structural** — `ToolContext.price_cache` and `news_cache` memoise within a session, so   even if the agent *did* re-call `get_price_data`, no second network request would occur.### 4.2 Persistent memory (5 marks)Keyed by ticker **and date** — `NVDA_2026-09-10.json`. Ticker alone would serve lastmonth's analysis as though it were current.

In [ ]:
cache = ResearchCache(ROOT / "task3_agentic" / "cache", max_age_hours=24)cache.clear(TICKER)   # Start clean so the demonstration is unambiguous.print("RUN 1 — cold")print("─" * 78)miss = cache.load(TICKER)print(f"Cache lookup : MISS ({miss.reason})")if result_3b["report"]:    path = cache.save(TICKER, result_3b["report"], metadata={        "tool_calls": len(tracer_3b.records),        "critique_rounds": result_3b["critique_rounds"],        "agents": ["AgentA", "AgentB"],    })    print(f"Saved        : {path.name}")print("\nRUN 2 — same ticker, same day")print("─" * 78)calls_before = len(tracer_3b.records)hit = cache.load(TICKER)if hit.hit:    print(f"Cache lookup : HIT ({hit.reason})")    print(f"Tool calls   : {len(tracer_3b.records) - calls_before}  ← zero; no tools re-run")    print(f"Cached at    : {hit.payload['cached_at']}")    print(f"Metadata     : {hit.payload['metadata']}")    print(f"\nLoaded report, first risk:")    print(f"  {hit.payload['report']['top_three_risks'][0]['risk'][:110]}")else:    print(f"Unexpected miss: {hit.reason}")print("\nBYPASS")print("─" * 78)forced = cache.load(TICKER, force_refresh=True)print(f"force_refresh=True → {'MISS' if not forced.hit else 'HIT'} ({forced.reason})")print("\nCache directory:")for entry in cache.list_entries():    print(" ", entry)

### 4.3 `agent_trace.jsonl` (5 marks)One JSON object per tool call: tool name, input arguments, output truncated to 200characters, and wall-clock duration.JSONL rather than a single JSON array so a run that crashes halfway still leaves a valid,parseable trace — which is exactly when you most want one. Truncation is *recorded* ratherthan hidden: `output_full_length` lets a reader distinguish "the tool returned 200characters" from "the tool returned 40KB and you are seeing the first 200".

In [ ]:
import pandas as pdrows = tracer_3b.load_all()print(f"{len(rows)} trace records at {TRACE_PATH.relative_to(ROOT)}\n")print("Sample record:")print(json.dumps(rows[0], indent=2)[:900])frame = pd.DataFrame(rows)print(f"\n{'─' * 78}\nPer-tool summary across all runs in this session:")summary = frame.groupby("tool").agg(    calls=("tool", "size"),    mean_ms=("duration_ms", "mean"),    max_ms=("duration_ms", "max"),    failures=("ok", lambda s: int((~s).sum())),).round(1).sort_values("calls", ascending=False)display(summary)print(f"\nTotal calls   : {len(frame)}")print(f"Total failures: {int((~frame['ok']).sum())}")print(f"Total time    : {frame['duration_ms'].sum() / 1000:.1f}s")print(f"Runs recorded : {frame['run_id'].nunique()}")display(frame[["seq", "agent", "tool", "duration_ms", "ok", "output"]].tail(12))

## 5 · Bonus — Streamlit trace dashboard (+5)`task3_agentic/dashboard.py` reads `agent_trace.jsonl` and renders the run visually:per-tool latency, the call sequence as a timeline, failure highlighting, and a run picker.Run it locally (VS Code on Windows):```bashpip install streamlit pandasstreamlit run task3_agentic/dashboard.py```From Colab you need a tunnel, since Colab cannot serve a port directly:```python!pip install -q streamlit!npm install -g localtunnel!streamlit run task3_agentic/dashboard.py &>/dev/null &!npx localtunnel --port 8501```Take a screenshot of the running dashboard and put it in `task3_agentic/README.md` —the bonus asks for visual evidence of a complete agent run trace.

In [ ]:
# Static preview of what the dashboard renders, so the notebook carries the# evidence even without a tunnel.import matplotlib.pyplot as pltframe = pd.DataFrame(tracer_3b.load_all())recent = frame[frame["run_id"] == frame["run_id"].iloc[-1]]fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))by_tool = recent.groupby("tool")["duration_ms"].mean().sort_values()ax1.barh(by_tool.index, by_tool.values, color="#2a6f97")ax1.set_xlabel("mean duration (ms)")ax1.set_title("Latency by tool", loc="left", fontweight="bold")ax1.grid(axis="x", alpha=0.3)colours = ["#0f7b4f" if ok_ else "#a32b2b" for ok_ in recent["ok"]]ax2.barh(range(len(recent)), recent["duration_ms"], color=colours)ax2.set_yticks(range(len(recent)))ax2.set_yticklabels([f"{r.seq}. {r.agent}:{r.tool}" for r in recent.itertuples()], fontsize=8)ax2.invert_yaxis()ax2.set_xlabel("duration (ms)")ax2.set_title("Call sequence (green = ok, red = failed)", loc="left", fontweight="bold")ax2.grid(axis="x", alpha=0.3)for ax in (ax1, ax2):    for spine in ("top", "right"):        ax.spines[spine].set_visible(False)plt.tight_layout(); plt.show()

## 6 · Rubric self-checkMechanical verification against every Task 3 criterion, so nothing is claimed that the rundoes not evidence.

In [ ]:
checks = []order_3a = [r.tool for r in tracer_3a.records]agents_seen = {r.agent for r in tracer_3b.records}b_used = {r.tool for r in tracer_3b.records if r.agent == "AgentB"}a_used = {r.tool for r in tracer_3b.records if r.agent == "AgentA"}all_rows = tracer_3b.load_all()checks += [    ("3A  All five tools callable", len({r.tool for r in smoke_tracer.records}) == 5,     f"{sorted({r.tool for r in smoke_tracer.records})}"),    ("3A  Autonomous tool selection", len(set(order_3a)) >= 3,     f"model chose {len(set(order_3a))} distinct tools, order {' → '.join(order_3a)}"),    ("3A  Observe-replan cycle visible", result_3a["iterations"] >= 2,     f"{result_3a['iterations']} agent→tools→agent iterations in the stream above"),    ("3A  Final report has all three sections", result_3a["report"] is not None,     "financial_health_summary / top_three_risks / hedge_strategy, schema-enforced"),    ("3A  Exactly three risks",     bool(result_3a["report"]) and len(result_3a["report"].top_three_risks) == 3,     "enforced by min_length=max_length=3"),    ("3A  Graceful tool failure", not bad_ticker["ok"] and bool(bad_ticker["suggestion"]),     "invalid ticker returned a suggestion, raised nothing"),    ("3B  Distinct roles, restricted tools", len(agents_seen) >= 2,     f"agents in trace: {sorted(agents_seen)}"),    ("3B  Restriction actually held",     not (b_used - set(AGENT_B_TOOLS)) and not (a_used - set(AGENT_A_TOOLS)),     f"A used {sorted(a_used)}, B used {sorted(b_used)} — zero violations"),    ("3B  Structured Pydantic handoff", result_3b["quant_brief"] is not None,     f"{type(result_3b['quant_brief']).__name__}, validated at the boundary"),    ("3B  Message trace visible", len(tracer_3b.records) > 0,     f"{len(tracer_3b.records)} calls printed inline with agent attribution"),    ("3B  Critique loop executed", result_3b["critique_rounds"] >= 1,     f"{result_3b['critique_rounds']} round, forced={result_3b['forced_critique']}"),    ("3B  End-to-end, no intervention", result_3b["report"] is not None,     "single graph.invoke() from query to final report"),    ("3C  Short-term memory", calls_after == calls_before,     "follow-up answered from context with zero new tool calls"),    ("3C  Persistent cache", hit.hit,     f"second lookup hit {cache.path_for(TICKER).name}"),    ("3C  agent_trace.jsonl present", TRACE_PATH.exists() and len(all_rows) > 0,     f"{len(all_rows)} records"),    ("3C  Trace has tool/inputs/output/duration",     all({"tool", "inputs", "output", "duration_ms"} <= set(r) for r in all_rows),     "every required field on every record"),]print(f"{'':4}{'Criterion':<42} {'':5} Evidence")print("─" * 116)for label, passed, detail in checks:    print(f"{'PASS' if passed else 'FAIL'}  {label:<42}       {detail[:62]}")print("─" * 116)print(f"{sum(c[1] for c in checks)}/{len(checks)} criteria evidenced by this run")

---## Design notes for the interview**Why the graph and not a while-loop.** A `while` loop around a tool-calling model wouldproduce the same behaviour and be shorter. What it would not give you is a *declared*control flow: with LangGraph the set of legal transitions is data, so "can Agent B reachthe price tools?" is answerable by reading the graph rather than by tracing every branch.On a system I have to defend line by line, that is worth the extra code.**Why `structured_call` instead of `.with_structured_output()`.** The prebuilt helperrelies on provider function-calling support, which free tiers implement inconsistently —it works on Groq's Llama 3.3 and silently degrades on several OpenRouter free models.Asking for JSON and validating with Pydantic works everywhere, and crucially it gives usthe validator error text to feed back on retry, which is what makes the repair convergerather than just re-rolling the dice.**Why `data_gaps` is required rather than optional.** This is the single most load-bearingschema decision in the task. Agent B cannot see Agent A's tools. If Agent A's volatilitycall fails and the field is optional, Agent A omits it, and Agent B reads the silence as"measured and unremarkable" — then writes a hedge recommendation on a number that was nevertaken. Making the declaration mandatory converts a silent failure into a visible one.**Known limitations.** Free-tier tool calling is the weakest link: Llama 3.3 occasionallyemits a malformed tool call, which surfaces as a wasted iteration. The iteration cap(`MAX_AGENT_ITERATIONS = 8`) bounds the damage but a production system would need retry-on-malformed-call. The critique loop is capped at one round; multi-round critique risksoscillation without a convergence criterion, and I did not have a principled one. Thehedge-strategy validator checks for *a* figure and *an* instrument, which catcheshand-waving but cannot verify the recommendation is sound — only a human or a backtest can.